In [5]:
import os
import sys
import torch
from matplotlib import pyplot as plt
from typing import Any, Dict

from lib import config_utils
from lib import data_utils
from lib import visutils
from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)

In [6]:
def get_dataloader_testdata(
    config_file_train: str,
    config_file_test: str,
    run_mode: str = 'test'
) -> torch.utils.data.dataloader.DataLoader:
    if not os.path.isfile(config_file_train):
        raise FileNotFoundError(f'Cannot find the configuration file used during training: {config_file_train}\n')

    if not os.path.isfile(config_file_test):
        raise FileNotFoundError(f'Cannot find the test configuration file: {config_file_test}\n')

    # Read the configuration file used during training
    config = config_utils.read_config(config_file_train)

    # Merge generic data settings (used during training) with test-specific data settings
    config_testdata = config_utils.read_config(config_file_test)
    config.data.update(config_testdata.data)
    if 'mask' in config_testdata:
        config.mask.update(config_testdata.mask)
    config.misc.run_mode = run_mode

    # Get the data loader
    dset = data_utils.get_dataset(config, phase=run_mode)
    dataloader = torch.utils.data.DataLoader(
        dataset=dset, batch_size=1, shuffle=False, num_workers=8, drop_last=False
    )
    
    return dataloader


def get_sample(
    dataloader: torch.utils.data.dataloader.DataLoader,
    sample_index: int
) -> Dict[str, Any]:
    
    batch = dataloader.dataset.__getitem__(sample_index)
    
    # Introduce the batch dimension (required for the forward pass)
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(0)
        elif isinstance(v, int):
            batch[k] = [v]

    return batch

## Imputation of satellite image time series with synthetic data gaps

<hr style="height: 2px;" />

**Settings**

In [7]:
# Default data and model settings (i.e., settings used during training)
config_file_train = 'configs/demo.yaml'

# Test-specific data settings
config_file_test = 'configs/config_earthnet2021_test_simulation.yaml'

# Model weights
checkpoint = 'checkpoints/utilise_earthnet2021.pth'
dataloader = get_dataloader_testdata(config_file_train, config_file_test)

**Get the data loader**

In [9]:
dataloader = get_dataloader_testdata(config_file_train, config_file_test)

**Instantiate U-TILISE**

In [11]:
for batch in dataloader: break
batch.keys()

In [22]:
batch["masks"].shape, batch["c_index_rgb"], batch["c_index_nir"], batch["cloud_mask"].shape

In [5]:
utilise = Imputation(config_file_train, method='utilise', checkpoint=checkpoint)

**Apply the model**

U-TILISE operates with a fixed temporal length of 10 images. I.e., U-TILISE processes time series consisting of at most 10 images in one shot, while it employs a sliding window approach to process sequences with more than 10 images.

In [6]:
# Get a data sample
batch = get_sample(dataloader, sample_index=269)

# Use U-TILISE to impute the satellite image time series
_, y_pred = utilise.impute_sample(batch)

# Let's visualize the sequences
brightness_factor = 5
indices_rgb = batch['c_index_rgb'][0]

fig = visutils.sequence2gallery(batch['x'][0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Input sequence with artificial data gaps', fontsize=7)

fig = visutils.sequence2gallery(batch['y'][0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Target sequence', fontsize=7)

fig = visutils.sequence2gallery(y_pred[0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('U-TILISE imputation', fontsize=7)

**Visualization of the attention masks**

We can visualize the attention masks only for satellite image time series whose temporal length complies with the temporal length of U-TILISE, as we cannot assemble the attention masks for the sliding window approach. For illustration purposes, we temporally trim longer sequences to a maximal temporal length of 10 images by specifying `t_start` and `t_end`. After temporal trimming, we apply U-TILISE to impute the chosen subsequence and visualize the corresponding attention masks.

In [7]:
# If the chosen sample comprises more than 10 images:
# Choose a subsequence consisting of at most 10 images to visualize the attention masks
t_start = 0
t_end = 10

# Use U-TILISE to impute the satellite image time series
batch, y_pred, att = utilise.impute_sample(batch, t_start=t_start, t_end=t_end, return_att=True)

Visualize the attention masks across all heads for imputing the `t_target`.th frame in the time series:

In [8]:
# Specify the index of the target frame (w.r.t. the temporally trimmed sequence)
t_target = 3

fig = visualize_att_for_target_t_across_heads(
    batch['x'], att, t_target, indices_rgb=indices_rgb, brightness_factor=brightness_factor
)

Visualize the attention masks of the `head`.th head across time:

In [9]:
head = 0

fig = visualize_att_for_one_head_across_time(
    batch['x'], att, head=head, indices_rgb=indices_rgb, brightness_factor=brightness_factor
)

**Comparison to non-learned baselines**

- `last`: copies the last observation
- `closest`: copies the temporally closest observation
- `linear_interpolation`: linearly interpolates between the most recent and the next observation

In [10]:
last = Imputation(config_file_train, method='trivial', mode='last')
closest = Imputation(config_file_train, method='trivial', mode='closest')
interp = Imputation(config_file_train, method='trivial', mode='linear_interpolation')

In [11]:
batch = get_sample(dataloader, sample_index=269)

# Perform imputation for every method
_, y_pred_utilise = utilise.impute_sample(batch)
_, y_pred_last = last.impute_sample(batch)
_, y_pred_closest = closest.impute_sample(batch)
_, y_pred_interp = interp.impute_sample(batch)

# Let's visualize the results
brightness_factor = 5
indices_rgb = batch['c_index_rgb'][0]

fig = visutils.sequence2gallery(batch['x'][0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Input sequence with artificial data gaps', fontsize=7)

fig = visutils.sequence2gallery(batch['y'][0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Target sequence', fontsize=7)

fig = visutils.sequence2gallery(y_pred_utilise[0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('U-TILISE imputation', fontsize=7)

fig = visutils.sequence2gallery(y_pred_last[0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Baseline: last', fontsize=7)

fig = visutils.sequence2gallery(y_pred_closest[0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Baseline: closest', fontsize=7)

fig = visutils.sequence2gallery(y_pred_interp[0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Baseline: linear interpolation', fontsize=7)

## Imputation of satellite image time series with real data gaps

<hr style="height: 2px;" />

**Settings**

In [12]:
# Default data and model settings (i.e., settings used during training)
config_file_train = 'configs/demo.yaml'

# Test-specific data settings
config_file_test = 'configs/config_earthnet2021_test.yaml'

# Model weights
checkpoint = 'checkpoints/utilise_earthnet2021.pth'

**Get the data loader**

In [13]:
dataloader = get_dataloader_testdata(config_file_train, config_file_test)

**Instantiate U-TILISE**

In [14]:
utilise = Imputation(config_file_train, method='utilise', checkpoint=checkpoint)

**Apply the model**

In [15]:
# Choose a data sample
batch = get_sample(dataloader, sample_index=294)

# Use U-TILISE to impute the satellite image time series
_, y_pred = utilise.impute_sample(batch)

# Let's visualize the sequences
brightness_factor = 5
indices_rgb = batch['c_index_rgb'][0]

fig = visutils.sequence2gallery(batch['y'][0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('Observed sequence', fontsize=7)

fig = visutils.sequence2gallery(y_pred[0], indices_rgb=indices_rgb, brightness_factor=brightness_factor)
plt.title('U-TILISE imputation', fontsize=7)

fig = visutils.sequence2gallery(batch['cloud_mask'][0], variable='binary_mask')
plt.title('Cloud masks', fontsize=7)